# Backpropagation from Scratch — AI Engineering Interview Prep

2-layer neural network in pure NumPy: forward pass, backward pass, gradient descent.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons, load_breast_cancer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

np.random.seed(42)

## 1. Activation Functions & Derivatives

In [ ]:
def sigmoid(z):
    return 1 / (1 + np.exp(-np.clip(z, -500, 500)))

def sigmoid_deriv(z):
    s = sigmoid(z)
    return s * (1 - s)

def relu(z):
    return np.maximum(0, z)

def relu_deriv(z):
    return (z > 0).astype(float)

def tanh(z):
    return np.tanh(z)

def tanh_deriv(z):
    return 1 - np.tanh(z)**2

x = np.linspace(-5, 5, 200)
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, (name, f, df) in zip(axes, [
    ('Sigmoid', sigmoid, sigmoid_deriv),
    ('ReLU',    relu,    relu_deriv),
    ('Tanh',    tanh,    tanh_deriv),
]):
    ax.plot(x, f(x), label='f(x)')
    ax.plot(x, df(x), '--', label="f'(x)")
    ax.set_title(name); ax.legend(); ax.axhline(0, color='k', lw=0.5)
plt.tight_layout()
plt.show()

## 2. Neural Network Class (2-Layer)

In [ ]:
class NeuralNetwork:
    def __init__(self, n_input, n_hidden, n_output, lr=0.01):
        self.lr = lr
        # He initialization for ReLU
        self.W1 = np.random.randn(n_input, n_hidden) * np.sqrt(2.0/n_input)
        self.b1 = np.zeros((1, n_hidden))
        self.W2 = np.random.randn(n_hidden, n_output) * np.sqrt(2.0/n_hidden)
        self.b2 = np.zeros((1, n_output))

    def forward(self, X):
        # Layer 1: linear + ReLU
        self.Z1 = X @ self.W1 + self.b1
        self.A1 = relu(self.Z1)
        # Layer 2: linear + Sigmoid (binary classification)
        self.Z2 = self.A1 @ self.W2 + self.b2
        self.A2 = sigmoid(self.Z2)
        return self.A2

    def compute_loss(self, y_hat, y):
        m = y.shape[0]
        y = y.reshape(-1, 1)
        # Binary cross-entropy
        eps = 1e-8
        loss = -np.mean(y * np.log(y_hat + eps) + (1-y) * np.log(1-y_hat + eps))
        return loss

    def backward(self, X, y):
        m = X.shape[0]
        y = y.reshape(-1, 1)

        # Output layer gradients
        dZ2 = self.A2 - y                          # dL/dZ2
        dW2 = (self.A1.T @ dZ2) / m
        db2 = dZ2.mean(axis=0, keepdims=True)

        # Hidden layer gradients (chain rule through ReLU)
        dA1 = dZ2 @ self.W2.T
        dZ1 = dA1 * relu_deriv(self.Z1)
        dW1 = (X.T @ dZ1) / m
        db1 = dZ1.mean(axis=0, keepdims=True)

        # Update weights
        self.W2 -= self.lr * dW2
        self.b2 -= self.lr * db2
        self.W1 -= self.lr * dW1
        self.b1 -= self.lr * db1

    def train(self, X, y, n_epochs=1000, verbose=True):
        losses = []
        for epoch in range(n_epochs):
            y_hat = self.forward(X)
            loss  = self.compute_loss(y_hat, y)
            self.backward(X, y)
            losses.append(loss)
            if verbose and epoch % 200 == 0:
                acc = ((y_hat.ravel() > 0.5) == y).mean()
                print(f"Epoch {epoch:5d}: loss={loss:.4f}, acc={acc:.4f}")
        return losses

    def predict(self, X):
        return (self.forward(X).ravel() > 0.5).astype(int)

print("NeuralNetwork class defined.")

## 3. Train on Moons Dataset

In [ ]:
X, y = make_moons(n_samples=1000, noise=0.1, random_state=42)
X = StandardScaler().fit_transform(X)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)

nn = NeuralNetwork(n_input=2, n_hidden=16, n_output=1, lr=0.05)
losses = nn.train(X_tr, y_tr, n_epochs=1000)

test_acc = (nn.predict(X_te) == y_te).mean()
print(f"\nTest accuracy: {test_acc:.4f}")

plt.figure(figsize=(10, 3))
plt.subplot(1,2,1)
plt.plot(losses)
plt.xlabel('Epoch'); plt.ylabel('Loss'); plt.title('Training Loss')

# Decision boundary
plt.subplot(1,2,2)
h = 0.02
x_min, x_max = X[:,0].min()-0.5, X[:,0].max()+0.5
y_min, y_max = X[:,1].min()-0.5, X[:,1].max()+0.5
xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))
Z = nn.forward(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
plt.contourf(xx, yy, Z, alpha=0.4, cmap='RdBu')
plt.scatter(X[:,0], X[:,1], c=y, cmap='RdBu', s=10, edgecolors='k', lw=0.3)
plt.title(f'Decision Boundary (acc={test_acc:.3f})')
plt.tight_layout()
plt.show()

## 4. Mini-Batch Gradient Descent

In [ ]:
class MiniBatchNN(NeuralNetwork):
    def train_minibatch(self, X, y, n_epochs=50, batch_size=32):
        m = X.shape[0]
        losses = []
        for epoch in range(n_epochs):
            # Shuffle
            idx = np.random.permutation(m)
            X_s, y_s = X[idx], y[idx]
            epoch_loss = 0
            for i in range(0, m, batch_size):
                Xb = X_s[i:i+batch_size]
                yb = y_s[i:i+batch_size]
                y_hat = self.forward(Xb)
                epoch_loss += self.compute_loss(y_hat, yb) * len(Xb)
                self.backward(Xb, yb)
            losses.append(epoch_loss / m)
            if epoch % 10 == 0:
                print(f"Epoch {epoch}: loss={losses[-1]:.4f}")
        return losses

nn_mb = MiniBatchNN(n_input=2, n_hidden=16, n_output=1, lr=0.1)
losses_mb = nn_mb.train_minibatch(X_tr, y_tr, n_epochs=50, batch_size=32)
print(f"Final test acc: {(nn_mb.predict(X_te) == y_te).mean():.4f}")

## Interview Tips

- **Backprop = chain rule**: dL/dW1 = dL/dA2 · dA2/dZ2 · dZ2/dA1 · dA1/dZ1 · dZ1/dW1.
- **Vanishing gradient**: sigmoid/tanh derivatives saturate near 0. ReLU avoids this (gradient = 1 or 0).
- **He init**: W ~ N(0, √(2/n_in)) for ReLU; Glorot init: N(0, √(2/(n_in+n_out))) for tanh.
- **Mini-batch SGD**: typically 32-256 samples. Adds noise that helps escape shallow minima.
- **Loss for binary classification**: binary cross-entropy. For multi-class: categorical cross-entropy + softmax.
- **Numerical gradient check**: compare analytical gradient vs (f(x+ε)-f(x-ε))/(2ε) — should match within 1e-5.